# Module 10 Guided Lab: SQL Joins and ABT Preparation

**BAN 6003: Data Management and Analytics Integration**

In this lab, we move from basic SQL exploration to SQL-based data integration and preliminary ABT construction.

The main idea for this module is:

> Before modeling, we need a table where each row matches the unit of analysis for the business question.

That table is often called an **Analytic Base Table**, or **ABT**.

## Lab Learning Goals

By the end of this lab, you should be able to:

1. Explain the purpose and structure of an Analytic Base Table.
2. Use SQL `INNER JOIN` and `LEFT JOIN`.
3. Use simple Common Table Expressions, or CTEs, with `WITH`.
4. Aggregate lower-level data before joining.
5. Build a preliminary one-row-per-entity ABT.
6. Read SQL results into Pandas.
7. Optional: observe how `ROW_NUMBER()` can help with deduplication.
8. Know the two ways to run SQL on your own project data (Section 11).

This lab assumes that you already completed Module 9 and can connect to the course Neon database.

## Business Scenario

Imagine you are helping an airline operations team prepare data for modeling or reporting.

The raw `flights` table has one row per flight. That is useful for operational detail, but it may not be the right table for every analysis.

Possible ABT units of analysis include:

- one row per flight
- one row per carrier-month
- one row per route
- one row per aircraft

In this lab, we will practice building preliminary ABTs using SQL.

## 0. Setup: Connect to Neon

This lab uses the same `.env` file as Module 9. If you completed Module 9 in this repository and environment, your `.env` file is ready. You do not need to change it.

`.env` is not saved to GitHub. If you opened a new Codespace or cloned your repository again:

1. Run the connection cell below. It creates an empty `.env` file in the top folder of your repository and stops with a message.
2. Open `.env` and paste the connection string from the Canvas page **"Module 9 Database Connection"** right after `DATABASE_URL=`. Save the file.
3. Run the connection cell again.

Never commit `.env`. The `.gitignore` file already tells Git to ignore it.


### Package setup note

If you are using **GitHub Codespaces**, the required packages should already be installed from `requirements.txt` when the Codespace is created. You usually do not need to run any `%pip install` command.

If you are running the repository on your **local computer** and an import fails, uncomment and run the `%pip install` line(s) in the setup cell below, then rerun the imports.



In [1]:
# Codespaces: these packages are installed from requirements.txt.
# Local only: if the imports below fail, uncomment and run this line:
# %pip install -q sqlalchemy psycopg2-binary "psycopg[binary]" python-dotenv pandas


In [2]:
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

from pathlib import Path

# Find the top folder of the repository (the folder that contains README.md).
repo_root = Path.cwd()
if not (repo_root / "README.md").exists() and (repo_root.parent / "README.md").exists():
    repo_root = repo_root.parent

# Create the .env file the first time. Git ignores this file, so it never goes to GitHub.
env_path = repo_root / ".env"
if not env_path.exists():
    env_path.write_text(
        '# Paste the connection string from the Canvas page "Module 9 Database Connection"\n'
        "# right after DATABASE_URL= (no spaces, no quotes). Then save this file.\n"
        "# Git ignores this file. Never commit it.\n"
        "DATABASE_URL=\n"
    )
    print("Created the file .env in the top folder of your repository.")

load_dotenv(env_path, override=True)
database_url = os.getenv("DATABASE_URL")

if not database_url:
    print("DATABASE_URL is empty. To fix it:")
    print("1. Open the file .env in the top folder of your repository (next to README.md).")
    print('2. Paste the connection string from the Canvas page "Module 9 Database Connection" right after DATABASE_URL=')
    print("3. Save .env and run this cell again.")
    raise RuntimeError("DATABASE_URL is not set. Follow the steps printed above.")

engine = create_engine(database_url)
print("Connection engine created. For safety, we do not print the connection string.")


Connection engine created. For safety, we do not print the connection string.


In [3]:
def run_query(query, engine):
    """Run a SQL query when it has been filled in; keep blank practice cells runnable."""
    if not query or not query.strip():
        print("This practice cell is waiting for your SQL. Add your query inside the triple quotes, then run it again.")
        return pd.DataFrame()
    return pd.read_sql(query, engine)


## 1. Quick Database Check

Before writing joins, check that the tables exist.

This query uses `information_schema.tables`, a system view that stores metadata about tables in the database.

In [4]:
query = '''
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
'''

run_query(query, engine)

,table_schema,table_name
0,public,airlines
1,public,airports
2,public,flights
3,public,planes
4,public,weather


We expect to see these tables:

- `flights`
- `airlines`
- `airports`
- `planes`
- `weather`

All course tables are in the default `public` schema, so you can use plain table names such as `flights`.

## 2. What Is an ABT?

An **Analytic Base Table** is a modeling-ready or analysis-ready table.

A good ABT has:

- a clear unit of analysis
- one row per entity or observation at that unit
- useful features or predictors
- a target variable if modeling is planned
- documented assumptions about how the table was built

The most important question is:

> What does one row mean?

## 3. SQL JOIN Basics

SQL joins combine rows from related tables.

The two core joins for this module are:

```sql
INNER JOIN
```

and

```sql
LEFT JOIN
```

An `INNER JOIN` keeps only matching rows.

A `LEFT JOIN` keeps all rows from the left table and adds matching information from the right table when available.

## 4. INNER JOIN Example: Flights with Airline Names

### Business question

Can we attach full airline names to flight records when the carrier code exists in both tables?

We join:

- `flights.carrier`
- `airlines.carrier`

In [5]:
query = '''
SELECT
    f.year,
    f.month,
    f.day,
    f.carrier,
    a.name AS airline_name,
    f.flight,
    f.origin,
    f.dest,
    f.dep_delay,
    f.arr_delay
FROM flights AS f
INNER JOIN airlines AS a
    ON f.carrier = a.carrier
LIMIT 10;
'''

run_query(query, engine)

,year,month,day,carrier,airline_name,flight,origin,dest,dep_delay,arr_delay
0,2013,1,1,UA,United Air Lines Inc.,1545,EWR,IAH,2.0,11.0
1,2013,1,1,UA,United Air Lines Inc.,1714,LGA,IAH,4.0,20.0
2,2013,1,1,AA,American Airlines Inc.,1141,JFK,MIA,2.0,33.0
3,2013,1,1,B6,JetBlue Airways,725,JFK,BQN,-1.0,-18.0
4,2013,1,1,DL,Delta Air Lines Inc.,461,LGA,ATL,-6.0,-25.0
5,2013,1,1,UA,United Air Lines Inc.,1696,EWR,ORD,-4.0,12.0
6,2013,1,1,B6,JetBlue Airways,507,EWR,FLL,-5.0,19.0
7,2013,1,1,EV,ExpressJet Airlines Inc.,5708,LGA,IAD,-3.0,-14.0
8,2013,1,1,B6,JetBlue Airways,79,JFK,MCO,-3.0,-8.0
9,2013,1,1,AA,American Airlines Inc.,301,LGA,ORD,-2.0,8.0


### SQL pattern

```sql
FROM flights AS f
INNER JOIN airlines AS a
    ON f.carrier = a.carrier
```

The aliases `f` and `a` make the query shorter and easier to read.

`AS airline_name` renames the output column so the result is clearer.

Every carrier code in `flights` also appears in `airlines`, so this inner join keeps all 336,776 flights. An inner join removes rows only when some keys have no match. Section 5 shows a join where that happens.

### Your Turn 1

Write an `INNER JOIN` that joins `flights` with `airports` using destination airport codes.

Use:

- `flights.dest`
- `airports.faa`

Return:

- flight date columns
- carrier
- flight number
- origin
- dest
- airport name as `dest_airport_name`
- arrival delay

Limit the result to 10 rows.

In [6]:
# Your Turn 1

query = '''
SELECT
    f.year,
    f.month,
    f.day,
    f.carrier,
    f.flight,
    f.origin,
    f.dest,
    a.name AS dest_airport_name,
    f.arr_delay
FROM flights AS f
INNER JOIN airports AS a
    ON f.dest = a.faa
LIMIT 10;
'''

run_query(query, engine)

,year,month,day,carrier,flight,origin,dest,dest_airport_name,arr_delay
0,2013,1,1,UA,1545,EWR,IAH,George Bush Intercontinental,11.0
1,2013,1,1,UA,1714,LGA,IAH,George Bush Intercontinental,20.0
2,2013,1,1,AA,1141,JFK,MIA,Miami Intl,33.0
3,2013,1,1,DL,461,LGA,ATL,Hartsfield Jackson Atlanta Intl,-25.0
4,2013,1,1,UA,1696,EWR,ORD,Chicago Ohare Intl,12.0
5,2013,1,1,B6,507,EWR,FLL,Fort Lauderdale Hollywood Intl,19.0
6,2013,1,1,EV,5708,LGA,IAD,Washington Dulles Intl,-14.0
7,2013,1,1,B6,79,JFK,MCO,Orlando Intl,-8.0
8,2013,1,1,AA,301,LGA,ORD,Chicago Ohare Intl,8.0
9,2013,1,1,B6,49,JFK,PBI,Palm Beach Intl,-2.0


## 5. LEFT JOIN Example: Keep All Flight Records

### Business question

What if we want to keep all flights, even if some destination airport information is missing?

A `LEFT JOIN` keeps all rows from the left table.

In [7]:
query = '''
SELECT
    f.year,
    f.month,
    f.day,
    f.carrier,
    f.flight,
    f.origin,
    f.dest,
    ap.name AS dest_airport_name,
    f.dep_delay,
    f.arr_delay
FROM flights AS f
LEFT JOIN airports AS ap
    ON f.dest = ap.faa
LIMIT 10;
'''

run_query(query, engine)

,year,month,day,carrier,flight,origin,dest,dest_airport_name,dep_delay,arr_delay
0,2013,1,1,UA,1545,EWR,IAH,George Bush Intercontinental,2.0,11.0
1,2013,1,1,UA,1714,LGA,IAH,George Bush Intercontinental,4.0,20.0
2,2013,1,1,AA,1141,JFK,MIA,Miami Intl,2.0,33.0
3,2013,1,1,B6,725,JFK,BQN,NaN,-1.0,-18.0
4,2013,1,1,DL,461,LGA,ATL,Hartsfield Jackson Atlanta Intl,-6.0,-25.0
5,2013,1,1,UA,1696,EWR,ORD,Chicago Ohare Intl,-4.0,12.0
6,2013,1,1,B6,507,EWR,FLL,Fort Lauderdale Hollywood Intl,-5.0,19.0
7,2013,1,1,EV,5708,LGA,IAD,Washington Dulles Intl,-3.0,-14.0
8,2013,1,1,B6,79,JFK,MCO,Orlando Intl,-3.0,-8.0
9,2013,1,1,AA,301,LGA,ORD,Chicago Ohare Intl,-2.0,8.0


### Row count check

A left join should usually keep the same number of rows as the left table, as long as the right table has unique keys.

Let's check.

In [8]:
query = '''
SELECT COUNT(*) AS flight_rows
FROM flights;
'''

run_query(query, engine)

,flight_rows
0,336776


In [9]:
query = '''
SELECT COUNT(*) AS joined_rows
FROM flights AS f
LEFT JOIN airports AS ap
    ON f.dest = ap.faa;
'''

run_query(query, engine)

,joined_rows
0,336776


Both counts are 336,776, so the left join kept every flight. Now run the **inner** join on the same key and count the rows.

In [10]:
query = '''
SELECT COUNT(*) AS inner_join_rows
FROM flights AS f
INNER JOIN airports AS ap
    ON f.dest = ap.faa;
'''

run_query(query, engine)


,inner_join_rows
0,329174


The inner join returns 329,174 rows: 7,602 fewer flights. Which destination codes have no match in `airports`?

After a left join, the right-table columns are empty for rows with no match. So `WHERE ap.faa IS NULL` keeps only the flights whose `dest` is not in `airports`.

In [11]:
query = '''
SELECT
    f.dest,
    COUNT(*) AS flight_count
FROM flights AS f
LEFT JOIN airports AS ap
    ON f.dest = ap.faa
WHERE ap.faa IS NULL
GROUP BY f.dest
ORDER BY flight_count DESC;
'''

run_query(query, engine)


,dest,flight_count
0,SJU,5819
1,BQN,896
2,STT,522
3,PSE,365


Four destination codes are not in the `airports` table: `SJU` (San Juan), `BQN` (Aguadilla), `STT` (St. Thomas), and `PSE` (Ponce). These airports are in Puerto Rico and the U.S. Virgin Islands. Their 5,819 + 896 + 522 + 365 = 7,602 flights are real flights:

- the **inner join** removes them;
- the **left join** keeps them and leaves `dest_airport_name` empty.

Neither result is wrong. Choose the join on purpose, and say which flights you removed.

Row count rules for a left join:

- more rows than the left table: the right table has duplicate keys;
- fewer rows than the left table: you probably used an inner join, or a `WHERE` condition on a right-table column removed rows.

### Your Turn 2

Use a `LEFT JOIN` to combine `flights` with `planes` using `tailnum`.

Return:

- flight date columns
- carrier
- flight number
- tailnum
- manufacturer
- model
- seats

Limit the result to 10 rows.

In [12]:
# Your Turn 2

query = '''
SELECT
    f.year,
    f.month,
    f.day,
    f.carrier,
    f.flight,
    f.tailnum,
    p.manufacturer,
    p.model,
    p.seats
FROM flights AS f
LEFT JOIN planes AS p
    ON f.tailnum = p.tailnum
LIMIT 10;
'''

run_query(query, engine)

,year,month,day,carrier,flight,tailnum,manufacturer,model,seats
0,2013,1,1,UA,1545,N14228,BOEING,737-824,149.0
1,2013,1,1,UA,1714,N24211,BOEING,737-824,149.0
2,2013,1,1,AA,1141,N619AA,BOEING,757-223,178.0
3,2013,1,1,B6,725,N804JB,AIRBUS,A320-232,200.0
4,2013,1,1,DL,461,N668DN,BOEING,757-232,178.0
5,2013,1,1,UA,1696,N39463,BOEING,737-924ER,191.0
6,2013,1,1,B6,507,N516JB,AIRBUS INDUSTRIE,A320-232,200.0
7,2013,1,1,EV,5708,N829AS,CANADAIR,CL-600-2B19,55.0
8,2013,1,1,B6,79,N593JB,AIRBUS,A320-232,200.0
9,2013,1,1,AA,301,N3ALAA,NaN,NaN,NaN


## 6. Simple CTEs with `WITH`

A CTE, or Common Table Expression, lets us create a named temporary result inside a SQL query.

The basic pattern is:

```sql
WITH temporary_table AS (
    SELECT ...
    FROM ...
)
SELECT *
FROM temporary_table;
```

CTEs make longer queries easier to read.

### Example: Create a clean flight subset first

Here, we use a CTE to keep only selected columns before joining.

In [13]:
query = '''
WITH flight_base AS (
    SELECT
        year,
        month,
        day,
        carrier,
        flight,
        origin,
        dest,
        dep_delay,
        arr_delay,
        distance,
        air_time
    FROM flights
)
SELECT
    fb.*,
    a.name AS airline_name
FROM flight_base AS fb
LEFT JOIN airlines AS a
    ON fb.carrier = a.carrier
LIMIT 10;
'''

run_query(query, engine)

,year,month,day,carrier,flight,origin,dest,dep_delay,arr_delay,distance,air_time,airline_name
0,2013,1,1,UA,1545,EWR,IAH,2.0,11.0,1400,227.0,United Air Lines Inc.
1,2013,1,1,UA,1714,LGA,IAH,4.0,20.0,1416,227.0,United Air Lines Inc.
2,2013,1,1,AA,1141,JFK,MIA,2.0,33.0,1089,160.0,American Airlines Inc.
3,2013,1,1,B6,725,JFK,BQN,-1.0,-18.0,1576,183.0,JetBlue Airways
4,2013,1,1,DL,461,LGA,ATL,-6.0,-25.0,762,116.0,Delta Air Lines Inc.
5,2013,1,1,UA,1696,EWR,ORD,-4.0,12.0,719,150.0,United Air Lines Inc.
6,2013,1,1,B6,507,EWR,FLL,-5.0,19.0,1065,158.0,JetBlue Airways
7,2013,1,1,EV,5708,LGA,IAD,-3.0,-14.0,229,53.0,ExpressJet Airlines Inc.
8,2013,1,1,B6,79,JFK,MCO,-3.0,-8.0,944,140.0,JetBlue Airways
9,2013,1,1,AA,301,LGA,ORD,-2.0,8.0,733,138.0,American Airlines Inc.


### Your Turn 3

Create a CTE called `jfk_flights` that selects flights where `origin = 'JFK'`.

Then join the CTE to `airlines` to add airline names.

Return 10 rows.

In [14]:
# Your Turn 3

query = '''
WITH jfk_flights AS (
    SELECT *
    FROM flights
    WHERE origin = 'JFK'
)
SELECT
    jf.*,
    a.name AS airline_name
FROM jfk_flights AS jf
LEFT JOIN airlines AS a
    ON jf.carrier = a.carrier
LIMIT 10;
'''

run_query(query, engine)

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,airline_name
0,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01T10:00:00Z,American Airlines Inc.
1,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01T10:00:00Z,JetBlue Airways
2,2013,1,1,557.0,600,-3.0,838.0,846,-8.0,B6,79,N593JB,JFK,MCO,140.0,944,6,0,2013-01-01T11:00:00Z,JetBlue Airways
3,2013,1,1,558.0,600,-2.0,849.0,851,-2.0,B6,49,N793JB,JFK,PBI,149.0,1028,6,0,2013-01-01T11:00:00Z,JetBlue Airways
4,2013,1,1,558.0,600,-2.0,853.0,856,-3.0,B6,71,N657JB,JFK,TPA,158.0,1005,6,0,2013-01-01T11:00:00Z,JetBlue Airways
5,2013,1,1,558.0,600,-2.0,924.0,917,7.0,UA,194,N29129,JFK,LAX,345.0,2475,6,0,2013-01-01T11:00:00Z,United Air Lines Inc.
6,2013,1,1,559.0,559,0.0,702.0,706,-4.0,B6,1806,N708JB,JFK,BOS,44.0,187,5,59,2013-01-01T10:00:00Z,JetBlue Airways
7,2013,1,1,606.0,610,-4.0,837.0,845,-8.0,DL,1743,N3739P,JFK,ATL,128.0,760,6,10,2013-01-01T11:00:00Z,Delta Air Lines Inc.
8,2013,1,1,611.0,600,11.0,945.0,931,14.0,UA,303,N532UA,JFK,SFO,366.0,2586,6,0,2013-01-01T11:00:00Z,United Air Lines Inc.
9,2013,1,1,613.0,610,3.0,925.0,921,4.0,B6,135,N635JB,JFK,RSW,175.0,1074,6,10,2013-01-01T11:00:00Z,JetBlue Airways


## 7. Aggregation Before Joining

This is one of the most important ABT ideas.

If your target table needs one row per carrier-month, you should aggregate the flight-level data to carrier-month before joining other carrier-month level information.

Do not join lower-level records in a way that accidentally duplicates rows.

### Example: Build a carrier-month delay summary

This creates one row per carrier-month.

As in Module 9, we write each average as `ROUND(AVG(column)::numeric, 2)::float`. The `::float` at the end gives Pandas normal numbers, not `Decimal` objects, so the columns work in plots and math.

In [15]:
query = '''
SELECT
    carrier,
    month,
    COUNT(*) AS flight_count,
    ROUND(AVG(dep_delay)::numeric, 2)::float AS avg_dep_delay,
    ROUND(AVG(arr_delay)::numeric, 2)::float AS avg_arr_delay,
    ROUND(AVG(distance)::numeric, 2)::float AS avg_distance
FROM flights
GROUP BY carrier, month
ORDER BY carrier, month;
'''

carrier_month_summary = run_query(query, engine)
carrier_month_summary.head()

,carrier,month,flight_count,avg_dep_delay,avg_arr_delay,avg_distance
0,9E,1,1573,16.88,10.21,476.35
1,9E,2,1459,16.49,8.28,467.89
2,9E,3,1627,13.41,2.03,476.49
3,9E,4,1511,13.57,5.47,492.47
4,9E,5,1462,22.67,10.42,505.49


Now one row means:

> one carrier in one month

This is no longer one row per flight.

### Add airline names after aggregation

Now we can join this carrier-month summary to the airlines table.

In [16]:
query = '''
WITH carrier_month AS (
    SELECT
        carrier,
        month,
        COUNT(*) AS flight_count,
        ROUND(AVG(dep_delay)::numeric, 2)::float AS avg_dep_delay,
        ROUND(AVG(arr_delay)::numeric, 2)::float AS avg_arr_delay,
        ROUND(AVG(distance)::numeric, 2)::float AS avg_distance
    FROM flights
    GROUP BY carrier, month
)
SELECT
    cm.*,
    a.name AS airline_name
FROM carrier_month AS cm
LEFT JOIN airlines AS a
    ON cm.carrier = a.carrier
ORDER BY cm.carrier, cm.month;
'''

carrier_month_abt = run_query(query, engine)
carrier_month_abt.head()

,carrier,month,flight_count,avg_dep_delay,avg_arr_delay,avg_distance,airline_name
0,9E,1,1573,16.88,10.21,476.35,Endeavor Air Inc.
1,9E,2,1459,16.49,8.28,467.89,Endeavor Air Inc.
2,9E,3,1627,13.41,2.03,476.49,Endeavor Air Inc.
3,9E,4,1511,13.57,5.47,492.47,Endeavor Air Inc.
4,9E,5,1462,22.67,10.42,505.49,Endeavor Air Inc.


### Light ABT check

For a carrier-month ABT, each carrier-month pair should appear once.

In [17]:
carrier_month_abt[["carrier", "month"]].duplicated().sum()

np.int64(0)

In [18]:
carrier_month_abt.isna().sum()

carrier          0
month            0
flight_count     0
avg_dep_delay    0
avg_arr_delay    0
avg_distance     0
airline_name     0
dtype: int64

### Your Turn 4

Build a route-level summary with one row per `origin` and `dest`.

Include:

- number of flights
- average departure delay
- average arrival delay
- average distance

Then add the destination airport name from the `airports` table as `dest_airport_name` (left join on `dest` = `faa`).

Use a CTE. Round each average with `ROUND(AVG(column)::numeric, 2)::float`.

In [19]:
# Your Turn 4

query = '''
WITH route_summary AS (
    SELECT
        origin,
        dest,
        COUNT(*) AS flight_count,
        ROUND(AVG(dep_delay)::numeric, 2)::float AS avg_dep_delay,
        ROUND(AVG(arr_delay)::numeric, 2)::float AS avg_arr_delay,
        ROUND(AVG(distance)::numeric, 2)::float AS avg_distance
    FROM flights
    GROUP BY origin, dest
)
SELECT
    r.*,
    ap.name AS dest_airport_name
FROM route_summary AS r
LEFT JOIN airports AS ap
    ON r.dest = ap.faa
ORDER BY r.origin, r.dest;
'''

route_abt = run_query(query, engine)
route_abt.head()

,origin,dest,flight_count,avg_dep_delay,avg_arr_delay,avg_distance,dest_airport_name
0,EWR,ALB,439,23.62,14.40,143.0,Albany Intl
1,EWR,ANC,8,12.88,-2.50,3370.0,Ted Stevens Anchorage Intl
2,EWR,ATL,5022,15.50,13.23,746.0,Hartsfield Jackson Atlanta Intl
3,EWR,AUS,968,11.51,-0.47,1504.0,Austin Bergstrom Intl
4,EWR,AVL,265,8.62,8.80,583.0,Asheville Regional Airport


In [20]:
# Your Turn 4 check
# Check whether each origin-dest pair appears once.
route_abt[["origin", "dest"]].duplicated().sum()

np.int64(0)

## 8. Preliminary ABT: One Row per Flight

Sometimes the ABT unit can remain one row per flight.

For example, if we want to predict whether a flight will arrive late, one row per flight may be appropriate.

Let's build a preliminary flight-level ABT that adds airline and airport context.

In [21]:
query = '''
WITH flight_base AS (
    SELECT
        year,
        month,
        day,
        carrier,
        flight,
        origin,
        dest,
        dep_delay,
        arr_delay,
        distance,
        air_time
    FROM flights
)
SELECT
    fb.*,
    a.name AS airline_name,
    dest_ap.name AS dest_airport_name,
    origin_ap.name AS origin_airport_name
FROM flight_base AS fb
LEFT JOIN airlines AS a
    ON fb.carrier = a.carrier
LEFT JOIN airports AS dest_ap
    ON fb.dest = dest_ap.faa
LEFT JOIN airports AS origin_ap
    ON fb.origin = origin_ap.faa
LIMIT 1000;
'''

flight_level_abt = run_query(query, engine)
flight_level_abt.head()

,year,month,day,carrier,flight,origin,dest,dep_delay,arr_delay,distance,air_time,airline_name,dest_airport_name,origin_airport_name
0,2013,1,1,UA,1545,EWR,IAH,2.0,11.0,1400,227.0,United Air Lines Inc.,George Bush Intercontinental,Newark Liberty Intl
1,2013,1,1,UA,1714,LGA,IAH,4.0,20.0,1416,227.0,United Air Lines Inc.,George Bush Intercontinental,La Guardia
2,2013,1,1,AA,1141,JFK,MIA,2.0,33.0,1089,160.0,American Airlines Inc.,Miami Intl,John F Kennedy Intl
3,2013,1,1,B6,725,JFK,BQN,-1.0,-18.0,1576,183.0,JetBlue Airways,NaN,John F Kennedy Intl
4,2013,1,1,DL,461,LGA,ATL,-6.0,-25.0,762,116.0,Delta Air Lines Inc.,Hartsfield Jackson Atlanta Intl,La Guardia


### Why limit to 1000 rows here?

`LIMIT 1000` keeps this demo fast. The full `flights` table has 336,776 rows. A full flight-level ABT saved as a CSV file is about 35 MB, and more joined columns make it larger. GitHub warns about files over 50 MB, and large files are slow to download and push.

If you choose a flight-level ABT in the Final Practice, you must filter it to a small subset. See the rule in Section 12.

In [22]:
flight_level_abt.shape

(1000, 14)

In [23]:
flight_level_abt.isna().sum()

year                    0
month                   0
day                     0
carrier                 0
flight                  0
origin                  0
dest                    0
dep_delay               4
arr_delay              11
distance                0
air_time               11
airline_name            0
dest_airport_name      33
origin_airport_name     0
dtype: int64

## 9. Data Leakage Warning

Data leakage happens when the ABT includes information that would not be available at the time of prediction.

For example, if you are trying to predict arrival delay before departure, using actual arrival delay as a feature would be leakage.

Leakage can make a model look very accurate during testing but fail in real use.

### Quick reflection

If your target is `arr_delay`, which columns should not be used as predictors?

Write a short answer below.

**Your answer:**  
If the target is `arr_delay`, we should not use `arr_delay` itself as a predictor. We should also avoid using `arr_time` and `air_time` because these values would not be available before the flight arrives. Using this information could cause data leakage and make the model appear more accurate than it actually is.

## 10. Optional Demo: `ROW_NUMBER()` to Keep One Row per Group

This section is optional. Advanced window functions are not required for core completion.

`ROW_NUMBER()` numbers the rows inside each group: 1, 2, 3, and so on. If you keep only the rows numbered 1, you keep exactly one row per group. This is a common way to remove duplicates, for example to keep only the most recent record for each customer.

Example pattern:

```sql
ROW_NUMBER() OVER (
    PARTITION BY key_column
    ORDER BY sorting_column DESC
)
```

`PARTITION BY` says what one group is. `ORDER BY` says which row in each group gets number 1.

Example: `flights` has thousands of rows per carrier. Keep each carrier's single most delayed departure.

In [24]:
query = '''
WITH ranked_flights AS (
    SELECT
        carrier,
        year,
        month,
        day,
        flight,
        origin,
        dest,
        dep_delay,
        ROW_NUMBER() OVER (
            PARTITION BY carrier
            ORDER BY dep_delay DESC NULLS LAST
        ) AS rn
    FROM flights
)
SELECT *
FROM ranked_flights
WHERE rn = 1
ORDER BY dep_delay DESC;
'''

run_query(query, engine)


,carrier,year,month,day,flight,origin,dest,dep_delay,rn
0,HA,2013,1,9,51,JFK,HNL,1301.0,1
1,MQ,2013,6,15,3535,JFK,CMH,1137.0,1
2,AA,2013,9,20,177,JFK,SFO,1014.0,1
3,DL,2013,4,10,2391,JFK,TPA,960.0,1
4,F9,2013,2,10,835,LGA,DEN,853.0,1
5,9E,2013,2,16,3798,JFK,CLT,747.0,1
6,VX,2013,7,7,187,EWR,SFO,653.0,1
7,FL,2013,9,12,350,LGA,ATL,602.0,1
8,EV,2013,12,5,4711,EWR,STL,548.0,1
9,B6,2013,1,16,517,EWR,MCO,502.0,1


The result has 16 rows, one per carrier, out of 336,776 flights.

- `PARTITION BY carrier` starts the numbering again for each carrier.
- `ORDER BY dep_delay DESC NULLS LAST` gives number 1 to the longest delay and puts missing delays last.
- If two rows tie, `ROW_NUMBER()` still keeps only one of them. Use `RANK()` if you want to keep ties.

Before you remove duplicates, check whether there are any. For example, `planes` has 3,322 rows and 3,322 different `tailnum` values, so it has no duplicate planes. Compare `COUNT(*)` with `COUNT(DISTINCT key)`.

Again, this is optional. The core expectation for Module 10 is joins, CTEs, aggregation before joining, and preliminary ABT construction.

## 11. SQL on Your Own Project Data: See Module 10 Part 2

So far, every query in this lab ran on the course Neon database. In your course project, you work with **your own data**, which is not in the course database. Project Milestone 2 requires at least one SQL query on your own project data. You choose one of two paths:

- **Path A, your own Neon database:** create a free Neon project, upload your project tables, and query them with the same SQL as in this lab.
- **Path B, DuckDB:** a small database engine that runs SQL directly on your Pandas DataFrames and CSV files, with no account and no connection string.

The notebook **`module10_part2_project_sql_options.ipynb`** in this repository teaches both paths step by step. It is not graded. Its cells marked `# COPY` can be copied into your project notebook unchanged.

A good project query supports your data work: for example, rebuild one aggregation or one join of your ABT in SQL, and compare the row count and keys with your Pandas result.

## 12. Final Practice: Build a Preliminary ABT

Choose one of the following units of analysis:

1. one row per carrier-month
2. one row per route
3. one row per flight (**a filter is required**, see below)

Write a SQL query that builds a preliminary ABT for your chosen unit.

Your ABT should include:

- the key columns that define one row
- at least three useful feature columns
- at least one joined descriptive field
- a clear row meaning

**Rule for a flight-level ABT:** add a `WHERE` clause that keeps one origin airport and one month, for example `WHERE f.origin = 'JFK' AND f.month = 1`. One airport in one month has about 7,000-11,000 rows (JFK in January has 9,161), and the CSV file is about 1 MB. Do not export all 336,776 flights. That file is about 35 MB or more, GitHub warns about files over 50 MB, and large files are slow to download and push. The export cell below will not save a table with more than 50,000 rows.

In [25]:
# Final Practice
# Build your preliminary ABT here.

query = '''
WITH route_summary AS (
    SELECT
        origin,
        dest,
        COUNT(*) AS flight_count,
        ROUND(AVG(dep_delay)::numeric, 2)::float AS avg_dep_delay,
        ROUND(AVG(arr_delay)::numeric, 2)::float AS avg_arr_delay,
        ROUND(AVG(distance)::numeric, 2)::float AS avg_distance
    FROM flights
    GROUP BY origin, dest
)
SELECT
    r.*,
    ap.name AS dest_airport_name
FROM route_summary AS r
LEFT JOIN airports AS ap
    ON r.dest = ap.faa
ORDER BY r.origin, r.dest;
'''

my_abt = run_query(query, engine)
my_abt.head()

,origin,dest,flight_count,avg_dep_delay,avg_arr_delay,avg_distance,dest_airport_name
0,EWR,ALB,439,23.62,14.40,143.0,Albany Intl
1,EWR,ANC,8,12.88,-2.50,3370.0,Ted Stevens Anchorage Intl
2,EWR,ATL,5022,15.50,13.23,746.0,Hartsfield Jackson Atlanta Intl
3,EWR,AUS,968,11.51,-0.47,1504.0,Austin Bergstrom Intl
4,EWR,AVL,265,8.62,8.80,583.0,Asheville Regional Airport


In [26]:
# Final Practice check 1
# Check the shape of your ABT.
my_abt.shape

(224, 7)

In [27]:
# Final Practice check 2
# Check missing values.
my_abt.isna().sum()

origin               0
dest                 0
flight_count         0
avg_dep_delay        1
avg_arr_delay        1
avg_distance         0
dest_airport_name    7
dtype: int64

In [28]:
# Final Practice check 3
# Check duplicate keys based on your chosen unit of analysis.
my_abt[["origin", "dest"]].duplicated().sum()

np.int64(0)

### Final Reflection

Write 5–7 sentences answering:

1. What is the unit of analysis in your ABT?
2. What keys define one row?
3. What tables did you join?
4. Did you aggregate before joining? Why or why not?
5. What checks did you run?
6. What would you still need to document before using this ABT in a project?

**Your reflection:**  
The unit of analysis in my ABT is one origin-destination route, meaning each row represents a unique route between two airports. The `origin` and `dest` columns define the keys for each row. I aggregated the `flights` table to the route level before joining it with the `airports` table to add destination airport names. Aggregating first allowed me to calculate the number of flights, average departure delay, average arrival delay, and average distance without creating duplicate route records. I checked the shape, missing values, and duplicate keys, and the results showed 224 rows, 7 columns, no duplicate route combinations, and some missing values. Before using this ABT in a project, I would need to document the missing values, how the averages were calculated, and what each row represents.

In [29]:
output_dir = repo_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

if my_abt.empty:
    print("Complete the Final Practice query before exporting.")
elif len(my_abt) > 50000:
    print(f"Your ABT has {len(my_abt):,} rows, so it was not saved.")
    print("For a flight-level ABT, add a WHERE filter (for example, one origin airport and one month) and run the query again.")
else:
    final_output_path = output_dir / "preliminary_abt_module10.csv"
    my_abt.to_csv(final_output_path, index=False)
    print(f"Saved {len(my_abt)} rows to: {final_output_path}")

Saved 224 rows to: /workspaces/ban6003-week-09-10-shanzayniazi/data/processed/preliminary_abt_module10.csv


## 13. Offline Assignment - Route-Month Operations ABT

A network planning team wants a stable table for comparing established route performance by month. Build `route_month_abt` with one row per `origin`-`dest`-`month` combination.

1. In a CTE, aggregate `flights` to route-month level with `flight_count`, `avg_dep_delay`, `avg_arr_delay`, and `avg_distance`. Round each average with `ROUND(AVG(column)::numeric, 2)::float`.
2. Use `HAVING` to keep route-month groups with at least 100 flights.
3. After aggregation, left join `airports` twice to add `origin_airport_name` and `dest_airport_name`.
4. Sort from highest to lowest average arrival delay.
5. Read the query into `route_month_abt`, check its shape, duplicate route-month keys, and missing values.
6. Save `data/processed/route_month_abt_module10.csv`.

In [30]:
# Offline Assignment - write and run your SQL query here.
# Suggested SQL structure:
# 1. WITH route_month AS (... GROUP BY origin, dest, month ...)
# 2. Apply HAVING COUNT(*) >= 100 inside the CTE.
# 3. SELECT the CTE columns and two airport-name aliases.
# 4. LEFT JOIN airports AS origin_ap on origin = faa.
# 5. LEFT JOIN airports AS dest_ap on dest = faa.
# 6. ORDER BY avg_arr_delay DESC.
query = '''
WITH route_month AS (
    SELECT
        origin,
        dest,
        month,
        COUNT(*) AS flight_count,
        ROUND(AVG(dep_delay)::numeric, 2)::float AS avg_dep_delay,
        ROUND(AVG(arr_delay)::numeric, 2)::float AS avg_arr_delay,
        ROUND(AVG(distance)::numeric, 2)::float AS avg_distance
    FROM flights
    GROUP BY origin, dest, month
    HAVING COUNT(*) >= 100
)
SELECT
    rm.*,
    origin_ap.name AS origin_airport_name,
    dest_ap.name AS dest_airport_name
FROM route_month AS rm
LEFT JOIN airports AS origin_ap
    ON rm.origin = origin_ap.faa
LEFT JOIN airports AS dest_ap
    ON rm.dest = dest_ap.faa
ORDER BY rm.avg_arr_delay DESC;
'''

route_month_abt = run_query(query, engine)
route_month_abt.head()


,origin,dest,month,flight_count,avg_dep_delay,avg_arr_delay,avg_distance,origin_airport_name,dest_airport_name
0,EWR,MCI,1,112,34.33,47.90,1092.0,Newark Liberty Intl,Kansas City Intl
1,EWR,RIC,6,143,46.67,45.64,277.0,Newark Liberty Intl,Richmond Intl
2,EWR,RIC,7,146,39.74,37.66,277.0,Newark Liberty Intl,Richmond Intl
3,EWR,RIC,12,145,35.89,37.17,277.0,Newark Liberty Intl,Richmond Intl
4,EWR,RIC,1,138,31.31,33.93,277.0,Newark Liberty Intl,Richmond Intl


In [31]:
if route_month_abt.empty:
    print("Complete the offline ABT query before validating and exporting.")
else:
    print("Shape:", route_month_abt.shape)
    print(
        "Duplicate route-month keys:",
        route_month_abt[["origin", "dest", "month"]].duplicated().sum()
    )
    print("Missing values by column:")
    display(route_month_abt.isna().sum())

    offline_output_path = output_dir / "route_month_abt_module10.csv"
    route_month_abt.to_csv(offline_output_path, index=False)
    print(f"Saved {len(route_month_abt)} rows to: {offline_output_path}")


Shape: (1041, 9)
Duplicate route-month keys: 0
Missing values by column:


origin                  0
dest                    0
month                   0
flight_count            0
avg_dep_delay           0
avg_arr_delay           0
avg_distance            0
origin_airport_name     0
dest_airport_name      18
dtype: int64

Saved 1041 rows to: /workspaces/ban6003-week-09-10-shanzayniazi/data/processed/route_month_abt_module10.csv


### ABT Recommendation

**Your response here:** Write 5-6 sentences explaining the unit of analysis and key, why aggregation occurs before the airport joins, what the 100-flight threshold does, what the validation evidence shows, and one limitation or leakage concern the team should document before modeling.

One row in `route_month_abt` represents one origin-destination route during one month. The `origin`, `dest`, and `month` columns define the key for each row. We aggregated the flight data before joining the airport tables to summarize flight performance and avoid creating duplicate route-month records. The 100-flight threshold limits the table to routes with at least 100 flights during a month, making comparisons more reliable. The validation checks showed 1,041 rows, no duplicate route-month combinations, and 18 missing destination airport names. Before using this ABT for modeling, the team should document these missing airport names and consider how they might affect the analysis.


## 14. Optional Challenge (Not Graded)

These challenges are for students who already know SQL from other courses and want more practice. They are **optional and not graded**. Skipping them does not affect your grade.

Each challenge uses SQL that goes beyond this module. If a function is new to you, look it up in the PostgreSQL documentation. An empty challenge cell still runs without an error, so **Restart and Run All** works whether or not you try them.

### Challenge 1: Add weather with a two-column key

Business question: do flights leave later when visibility at the origin airport is low?

The `weather` table has one row per origin airport per hour. Its key has **two** columns: `origin` and `time_hour`.

1. Check that the key is unique: compare `COUNT(*)` with `COUNT(DISTINCT (origin, time_hour))` in `weather`.
2. Left join `flights` to `weather` on **both** columns: `ON f.origin = w.origin AND f.time_hour = w.time_hour`.
3. Put each flight in a visibility group from `visib` (miles): `under 1 mile`, `1 to 5 miles` (1 or more, under 5), `5 miles or more`, and `no weather record`.
4. Return one row per group with `flight_count` and `avg_dep_delay`.

**Hints:** use a `CASE WHEN` expression for the groups; test `w.visib IS NULL` first. A CTE for the joined rows keeps the query readable.

**Check your result:** the key check gives two equal numbers, so the join cannot add rows. The four `flight_count` values add up to 336,776, and 1,556 flights have no weather record. Remember that this shows an association, not a cause.

In [32]:
# Challenge 1 (optional)
query = '''

'''

run_query(query, engine)


This practice cell is waiting for your SQL. Add your query inside the triple quotes, then run it again.


""


### Challenge 2: Which flights have no aircraft record?

Business question: if an ABT uses aircraft features from `planes`, which carriers would lose the most flights?

Find the flights whose `tailnum` has no match in `planes` (Section 5 shows the `LEFT JOIN ... WHERE ... IS NULL` pattern). Return one row per carrier with `flights_without_plane_record` and `missing_tailnum` (how many of those flights have no `tailnum` at all). Sort from the most to the fewest flights without a record.

**Hints:** `COUNT(*) FILTER (WHERE f.tailnum IS NULL)` counts only the rows that meet the condition. It does the same job as `SUM(CASE WHEN ... THEN 1 ELSE 0 END)`.

**Check your result:** the `flights_without_plane_record` values add up to 52,606. Then think: for the top two carriers, most of these flights do have a `tailnum`. What does that say about the `planes` table, and what would an inner join with `planes` do to a carrier comparison?

In [33]:
# Challenge 2 (optional)
query = '''

'''

run_query(query, engine)


This practice cell is waiting for your SQL. Add your query inside the triple quotes, then run it again.


""


### Challenge 3: A carrier-month ABT with a next-month target

Business question: can this month's performance help predict next month's arrival delay?

Build a carrier-month ABT with this month's `flight_count`, `avg_dep_delay`, and `avg_arr_delay` as features, and `next_month_avg_arr_delay` as the target. Round the averages as in Section 7.

**Hints:** aggregate in a CTE first, then use `LEAD(avg_arr_delay) OVER (PARTITION BY carrier ORDER BY month)`. `LEAD` takes the value from the next row in the same carrier.

**Check your result:** 185 rows, and 16 of them have an empty target (the last month of each carrier). Then answer two questions in a comment:

1. Carrier `OO` flew in only 5 of the 12 months. For `OO`, is the next row always the next calendar month? How would you check, and how would you fix it?
2. Why is it fine to use this month's `avg_arr_delay` as a feature here, when Section 9 warned against using `arr_delay` to predict `arr_delay`?

In [34]:
# Challenge 3 (optional)
query = '''

'''

next_month_abt = run_query(query, engine)
next_month_abt.head(12)


This practice cell is waiting for your SQL. Add your query inside the triple quotes, then run it again.


""


## 15. Save Your Work

Before submitting:

1. Complete all Your Turn, final-practice, and offline-assignment sections. The Optional Challenge in Section 14 is not required.
2. Complete all three written answers: the data leakage quick reflection (Section 9), the Final Reflection (Section 12), and the ABT Recommendation (Section 13).
3. Restart the kernel and run all cells from top to bottom.
4. Confirm both required Module 10 CSV files exist in `data/processed`: `preliminary_abt_module10.csv` and `route_month_abt_module10.csv`.
5. Confirm `.env` is not tracked by Git.
6. Save the notebook, then commit and push your work.

```bash
git add .
git commit -m "Complete Module 10 SQL joins and ABT lab"
git push
```